# Pratica — Lezione 8: riassumi la trascrizione con un modello gratuito

Nel progetto reale "AI Meeting Companion" di IBM, dopo aver trascritto l'audio con
Whisper, il testo viene passato a un LLM con un prompt del tipo "*List the key points with
details from the context*" per estrarre i punti chiave della riunione. Il materiale originale
usa un modello a pagamento di IBM watsonx per questo passaggio. In questa pratica risolverai
lo stesso caso d'uso — trascrivere e riassumere una riunione — usando un modello di riassunto
100% gratuito di Hugging Face.

## Esercizio — Completa `summarize_meeting()`

Hai già pronta la trascrizione dell'audio di esempio (come nel notebook di lezione). Il tuo
compito è completare la funzione `summarize_meeting()` affinché, dato il testo trascritto,
restituisca un riassunto con i punti chiave, usando un pipeline di riassunto di Hugging Face
(per esempio `sshleifer/distilbart-cnn-12-6`, una versione più leggera di
`facebook/bart-large-cnn`).

In [ ]:
!pip install -q transformers torch requests

In [ ]:
import requests
from transformers import pipeline

# 1) Scarichiamo lo stesso audio di esempio della lezione
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-GPXX04C6EN/Testing%20speech%20to%20text.mp3"
response = requests.get(url)
audio_file_path = "downloaded_audio.mp3"
with open(audio_file_path, "wb") as file:
    file.write(response.content)

# 2) Trascriviamo l'audio con Whisper, come nel notebook di lezione
asr_pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-tiny.en",
    chunk_length_s=30,
)
transcript_txt = asr_pipe(audio_file_path, batch_size=8)["text"]
print("Trascrizione:")
print(transcript_txt)

In [ ]:
# TODO 1: crea un pipeline di riassunto ("summarization") con un modello gratuito
#         di Hugging Face, per esempio "sshleifer/distilbart-cnn-12-6"
summarizer = None  # <-- sostituisci questo

def summarize_meeting(text):
    # TODO 2: usa il summarizer per generare un riassunto del testo.
    #         Suggerimento: summarizer(text, max_length=60, min_length=15, do_sample=False)
    #         L'output è una lista di dizionari con la chiave "summary_text".
    #         Restituisci la stringa del riassunto.
    pass  # <-- sostituisci questo

# Decommenta per provare una volta completati i TODO:
# print(summarize_meeting(transcript_txt))

<details>
<summary>💡 Vedi soluzione</summary>

```python
from transformers import pipeline

summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")

def summarize_meeting(text):
    resultado = summarizer(text, max_length=60, min_length=15, do_sample=False)
    return resultado[0]["summary_text"]

print(summarize_meeting(transcript_txt))
```

Nota: se il testo trascritto è molto corto, alcuni modelli di riassunto possono fallire o
restituire quasi lo stesso testo — è normale, i modelli di riassunto sono pensati per testi
più lunghi, come una riunione completa. Puoi anche provare con `facebook/bart-large-cnn`
al posto di `sshleifer/distilbart-cnn-12-6`: è più preciso, ma più lento e pesante.
</details>

## Riferimento: così si fa con l'LLM a pagamento del materiale originale (watsonx)

Il progetto reale `AIMeetingCompanion_Project/speech_analyzer.py` concatena la trascrizione
con un LLM di IBM watsonx usando LangChain:

```python
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from ibm_watson_machine_learning.foundation_models.extensions.langchain import WatsonxLLM

temp = '''
<s><<SYS>>
List the key points with details from the context:
[INST] The context : {context} [/INST]
<</SYS>>
'''
pt = PromptTemplate(input_variables=["context"], template=temp)
prompt_to_LLAMA2 = LLMChain(llm=llm, prompt=pt)  # llm = WatsonxLLM(...)

result = prompt_to_LLAMA2.run(transcript_txt)
```

Questa via richiede credenziali a pagamento di IBM watsonx; per questo, in questa pratica hai
risolto lo stesso caso d'uso con un modello di riassunto gratuito di Hugging Face.

Torna al [notebook di lezione](../notebooks_clase/clase08_whisper_voz_IT.ipynb) o ripassa il
[Capitolo 12 del manuale](../manual/12_whisper_voz_IT.md).